# Step 2: Set Up Your Geospatial Notebook

This notebook loads spatial data (field boundaries) and soil data, verifies CRS alignment, and prepares data for geospatial analysis.

In [ ]:
# Import required libraries
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

print("Libraries imported successfully")

## 1. Load Field Boundaries (GeoJSON)

In [ ]:
# Load field boundaries from GeoJSON
fields = gpd.read_file('../data/michigan_fields.geojson')

print(f"Loaded {len(fields)} fields")
print(f"Columns: {list(fields.columns)}")
print(f"\nCRS: {fields.crs}")
fields.head()

## 2. Load Soil Data (SSURGO)

In [ ]:
# Load soil data from SSURGO (downloaded via NRCS API)
soil = pd.read_csv('../data/michigan_soil_ssurgo.csv')

print(f"Loaded {len(soil)} soil records")
print(f"Fields with soil data: {soil['field_id'].nunique()}")
print(f"\nColumns: {list(soil.columns)}")
soil.head()

## 3. Check CRS - Verify Alignment

In [ ]:
# Check CRS of field boundaries
print("=== CRS CHECK ===")
print(f"Field boundaries CRS: {fields.crs}")
print(f"Field boundaries CRS (EPSG): {fields.crs.to_epsg()}")

# Soil data is attribute data (CSV) - no spatial CRS
# When joined to fields, the combined layer inherits field CRS
print(f"\nSoil data: Attribute table (CSV) - no spatial CRS")
print("Note: Soil data will be joined to field boundaries using 'field_id'")
print("      The combined layer will inherit field boundary CRS (EPSG:4326)")

## 4. Join Soil Data to Field Boundaries

In [ ]:
# Get dominant soil per field (first row per field_id, sorted by comppct_r)
dominant_soil = soil.sort_values('comppct_r', ascending=False).groupby('field_id').first().reset_index()
dominant_soil = dominant_soil[['field_id', 'mukey', 'muname', 'compname', 'comppct_r', 
                                'drainagecl', 'om_r', 'ph1to1h2o_r', 'claytotal_r', 'sandtotal_r']]

# Join to fields
fields_with_soil = fields.merge(dominant_soil, on='field_id', how='left')

print(f"Fields with soil joined: {len(fields_with_soil)}")
print(f"CRS after join: {fields_with_soil.crs}")
fields_with_soil[['field_id', 'county', 'crop_name', 'compname', 'om_r', 'ph1to1h2o_r']].head()

## 5. Summary - CRS Verification

In [ ]:
# Final CRS verification
print("=== FINAL CRS VERIFICATION ===")
print(f"Combined layer CRS: {fields_with_soil.crs}")
print(f"CRS is WGS84 (EPSG:4326): {fields_with_soil.crs.to_epsg() == 4326}")

# Check CRS type
print(f"\nCRS type: {fields_with_soil.crs.geodetic_crs.name if hasattr(fields_with_soil.crs, 'geodetic_crs') else 'Geographic'}")
print("\n✅ Both layers use the same CRS (EPSG:4326)")
print("   - Field boundaries: EPSG:4326 (WGS84)")
print("   - Soil data: Joined as attributes, inherits field CRS")

## 6. Quick Visualization

In [ ]:
# Simple visualization of fields with soil data
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

fields_with_soil.plot(column='om_r', ax=ax, legend=True, 
                       legend_kwds={'label': 'Organic Matter (%)'},
                       cmap='Greens', edgecolor='black')

ax.set_title('Field Boundaries - Organic Matter Content')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

plt.tight_layout()
plt.show()

## Step 3: Create an Integrated Map (The "2 Layer" Task)

Plot field boundaries as the base layer and overlay a second spatial dataset:
- **Base layer**: Field boundaries (polygons)
- **Overlay**: Color fields by dominant soil type

In [ ]:
# Integrated Map: Fields colored by Dominant Soil Type
fig, ax = plt.subplots(1, 1, figsize=(12, 10))

# Plot fields colored by dominant soil type
fields_with_soil.plot(column='compname', 
                      ax=ax,
                      legend=True,
                      legend_kwds={'title': 'Dominant Soil Type', 'loc': 'lower left'},
                      cmap='Set2', 
                      edgecolor='black',
                      linewidth=1.5)

# Add field labels
for idx, row in fields_with_soil.iterrows():
    centroid = row.geometry.centroid
    ax.annotate(f"{row['compname']}\n({row['county']})", 
                xy=(centroid.x, centroid.y),
                ha='center', va='center',
                fontsize=8,
                fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

ax.set_title('Michigan Fields - Dominant Soil Types\n(Base: Field Boundaries | Overlay: Soil Type)', 
             fontsize=14, fontweight='bold')
ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)

plt.tight_layout()
plt.show()

## Alternative: Color by Drainage Class

In [ ]:
# Integrated Map: Fields colored by Drainage Class
fig, ax = plt.subplots(1, 1, figsize=(12, 10))

# Plot fields colored by drainage class
fields_with_soil.plot(column='drainagecl', 
                      ax=ax,
                      legend=True,
                      legend_kwds={'title': 'Drainage Class', 'loc': 'lower left'},
                      cmap='RdYlBu', 
                      edgecolor='black',
                      linewidth=1.5)

# Add field labels
for idx, row in fields_with_soil.iterrows():
    centroid = row.geometry.centroid
    ax.annotate(f"{row['drainagecl']}\n{row['area_acres']:.1f} ac", 
                xy=(centroid.x, centroid.y),
                ha='center', va='center',
                fontsize=8,
                fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

ax.set_title('Michigan Fields - Drainage Class\n(Base: Field Boundaries | Overlay: Drainage)', 
             fontsize=14, fontweight='bold')
ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)

plt.tight_layout()
plt.show()

## Summary: Two-Layer Integration

In [ ]:
# Summary table: Field, Soil Type, and Drainage
summary = fields_with_soil[['field_id', 'county', 'area_acres', 'crop_name', 
                            'compname', 'drainagecl', 'om_r', 'ph1to1h2o_r']].copy()
summary = summary.rename(columns={
    'field_id': 'Field ID',
    'county': 'County',
    'area_acres': 'Area (acres)',
    'crop_name': 'Crop',
    'compname': 'Soil Type',
    'drainagecl': 'Drainage Class',
    'om_r': 'Organic Matter (%)',
    'ph1to1h2o_r': 'pH'
})
print("=== Field-Soil Summary ===")
print(summary.to_string(index=False))

## Individual Field Maps

Each field is shown separately with its boundary, location context, and soil properties.

In [ ]:
for idx, field in fields_with_soil.iterrows():
    fig, ax = plt.subplots(1, 1, figsize=(12, 10))
    
    # Get bounds with padding
    bounds = field.geometry.bounds
    padding = 0.02
    
    # Plot all fields in light gray (context)
    fields_with_soil.plot(ax=ax, facecolor='lightgray', edgecolor='gray', alpha=0.3)
    
    # Plot this field highlighted
    gpd.GeoSeries([field.geometry]).plot(ax=ax, facecolor='lightgreen', 
                                         edgecolor='black', linewidth=2)
    
    # Set axis limits AFTER plotting to lock the zoom
    ax.set_xlim(bounds[0] - padding, bounds[2] + padding)
    ax.set_ylim(bounds[1] - padding, bounds[3] + padding)
    
    # Add field label at centroid

## Interpretation

The maps reveal significant soil variability across the 5 clustered Michigan fields in southeast Michigan. The fields contain diverse soil types including Cadmus, Selfridge, Pella, and Corunna with varying drainage characteristics from moderately well drained to poorly drained. Organic matter ranges from 0.53% to 5.5% and pH ranges from 5.8 to 7.0. This soil variability across the clustered fields provides excellent opportunities for precision agriculture management zone delineation based on these soil characteristics.

In [ ]:
# Updated Field Summary with Real Data
print("=== Updated Field Summary (5 Clustered Fields with Irregular Shapes) ===")
for idx, row in fields_with_soil.iterrows():
    compname = row['compname'] if pd.notna(row['compname']) else 'N/A'
    om = row['om_r'] if pd.notna(row['om_r']) else 'N/A'
    ph = row['ph1to1h2o_r'] if pd.notna(row['ph1to1h2o_r']) else 'N/A'
    drainage = row['drainagecl'] if pd.notna(row['drainagecl']) else 'N/A'
    print(f"Field {idx+1}: {row['county']} | {row['crop_name']} | Soil: {compname} | Drainage: {drainage} | OM: {om}% | pH: {ph}")
print(f"\nTotal: {len(fields_with_soil)} fields")

## Step 5: Design Your Dashboard Map

This is the final, polished spatial visualization for dashboard use. The map shows all 5 fields with their dominant soil types. Field names are positioned at the edge of each polygon to avoid blocking the boundaries. Detailed field data is provided in the description section below the map.

In [ ]:
# Create polished dashboard map with edge labels and description
fig, ax = plt.subplots(1, 1, figsize=(12, 9))
fig.patch.set_facecolor('white')

# Plot fields colored by soil type
fields_with_soil.plot(column='compname', 
                     ax=ax,
                     legend=True,
                     legend_kwds={'title': 'Dominant Soil Type', 
                                 'loc': 'upper left',
                                 'fontsize': 9,
                                 'title_fontsize': 10},
                     cmap='Set2', 
                     edgecolor='black', 
                     linewidth=2,
                     alpha=0.85)

# Add field name labels at EDGE of each field (not blocking boundary)
for idx, row in fields_with_soil.iterrows():
    geom = row.geometry
    bounds = geom.bounds
    label_x = bounds[0] + 0.003
    label_y = bounds[1] + 0.003
    ax.annotate(f"{row['county']}", 
                xy=(label_x, label_y),
                ha='left', va='bottom',
                fontsize=10,
                fontweight='bold',
                color='darkblue',
                bbox=dict(boxstyle='round,pad=0.2', 
                         facecolor='white', 
                         edgecolor='darkblue',
                         alpha=0.9),
                zorder=10)

# Title
ax.set_title("Michigan Agricultural Fields - Soil Analysis Dashboard", 
             fontsize=16, fontweight='bold', pad=15)

# Axis labels
ax.set_xlabel('Longitude (°W)', fontsize=10)
ax.set_ylabel('Latitude (°N)', fontsize=10)

# Add grid
ax.grid(True, linestyle='--', alpha=0.3)

# Set aspect ratio
ax.set_aspect('equal')

# Adjust layout for description section
plt.tight_layout()
fig.subplots_adjust(bottom=0.28)

# Add description section at bottom
description_ax = fig.add_axes([0.08, 0.02, 0.84, 0.22])
description_ax.axis('off')

# Field list
field_list_text = "Fields: "
for idx, row in fields_with_soil.iterrows():
    field_list_text += f"✦ {row['county']} ({row['crop_name']}, {row['area_acres']:.0f} ac)  "
    if idx < len(fields_with_soil) - 1:
        field_list_text += "\n              "

description_ax.text(0.02, 0.95, field_list_text, 
                    transform=description_ax.transAxes,
                    fontsize=9, va='top', fontfamily='monospace')

# Add explanatory paragraph
paragraph = (
    "The map displays 5 agricultural fields in southeast Michigan, colored by dominant soil type. "
    "The western cluster (Lenawee, Jackson) shows Cadmus and Corunna soils, indicating moderately well "
    "to poorly drained conditions typically suited for corn production. The eastern fields (Monroe, Wayne, Washtenaw) "
    "display Selfridge and Pella soils with higher organic matter (up to 5.5%), preferred for soybean cultivation. "
    "This soil-based coloring reveals distinct drainage patterns across the field cluster, guiding precision "
    "agriculture management decisions."
)

description_ax.text(0.02, 0.35, paragraph, 
                    transform=description_ax.transAxes,
                    fontsize=10, va='top', wrap=True)

# Save to output
plt.savefig('output/dashboard_assets/field_spatial_map.png', 
            dpi=150, bbox_inches='tight', 
            facecolor='white', edgecolor='none')
print("Dashboard map saved to output/dashboard_assets/field_spatial_map.png")
plt.show()